# YouTube Subtitle Harvester (yt-dlp)

This notebook downloads and processes YouTube subtitles/captions (both manual and auto-generated) using `yt-dlp`.

### Features:
- Inspect available subtitles and automatic captions across languages.
- Download single video or entire playlist subtitles in `.vtt`, `.srt`, or clean text format.
- Parse timestamped VTT/SRT into clean, deduplicated text.
- Native support for Nepali (`ne`), English (`en`), and other low-resource / Indic languages.

## 1. Install Dependencies

In [1]:
# Install yt-dlp if not already installed in your environment
!pip install --upgrade yt-dlp

zsh:1: command not found: pip


## 2. Utility Functions for Subtitle Extraction & Cleaning

In [2]:
import os
import re
from pathlib import Path
import yt_dlp

def clean_vtt_text(vtt_path):
    """
    Cleans a WebVTT file by stripping header metadata, timing lines,
    cue positioning, and inline timestamp tags (e.g. <00:00:01.200>).
    Deduplicates consecutive identical phrases.
    """
    with open(vtt_path, 'r', encoding='utf-8') as f:
        content = f.read()

    lines = content.splitlines()
    cleaned_lines = []
    prev_line = ""

    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith("WEBVTT") or line.startswith("Kind:") or line.startswith("Language:"):
            continue
        if line.startswith("NOTE") or "-->" in line:
            continue

        # Remove HTML-like tags and inline timing tokens: <00:00:01.200><c> text </c>
        text = re.sub(r"<[^>]+>", "", line).strip()
        if not text:
            continue

        # Deduplicate consecutive identical lines
        if text != prev_line:
            cleaned_lines.append(text)
            prev_line = text

    return "\n".join(cleaned_lines)

def list_available_captions(url):
    """
    Inspect available manual subtitles and automatic captions for a YouTube video.
    """
    ydl_opts = {'skip_download': True, 'quiet': True}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=False)
        title = info.get('title', 'Unknown Title')
        manual_subs = list(info.get('subtitles', {}).keys())
        auto_subs = list(info.get('automatic_captions', {}).keys())
        print(f"Title: {title}")
        print(f"Manual subtitles available ({len(manual_subs)}): {manual_subs[:10]}")
        print(f"Auto-captions available ({len(auto_subs)}): {auto_subs[:10]} ...")
        return manual_subs, auto_subs

def download_subtitles(url, languages=["ne", "en"], output_dir=".", clean_text=True):
    """
    Download subtitles (preferring manual, falling back to auto-generated).
    Optionally converts the .vtt file to a clean .txt transcript.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    out_tmpl = str(Path(output_dir) / "%(title)s_%(id)s.%(ext)s")

    ydl_opts = {
        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': languages,
        'subtitlesformat': 'vtt',
        'outtmpl': out_tmpl,
        'quiet': False,
        'ignoreerrors': True
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        video_id = info.get('id', '')
        video_title = info.get('title', '')

    downloaded_files = []
    for p in Path(output_dir).glob(f"*{video_id}*.vtt"):
        downloaded_files.append(p)
        if clean_text:
            cleaned_content = clean_vtt_text(p)
            txt_path = p.with_suffix(".txt")
            with open(txt_path, 'w', encoding='utf-8') as f:
                f.write(f"Video Title: {video_title}\n")
                f.write(f"Video URL: {url}\n")
                f.write("-" * 40 + "\n")
                f.write(cleaned_content)
            print(f"Clean transcript saved to: {txt_path}")

    return downloaded_files

## 3. Test with Video: `https://www.youtube.com/watch?v=JgNZQ4ZTpZQ`

In [3]:
video_url = "https://www.youtube.com/watch?v=JgNZQ4ZTpZQ"

# 1. Check available captions
manual, auto = list_available_captions(video_url)

Title: पत्रकारलाई ‘ठीक्क पर्‍यो’ भन्नेहरूले पनि बुझ्नुपर्ने चुरो कुरा | Press Freedom Explained Nepal |
Manual subtitles available (0): []
Auto-captions available (157): ['ab', 'aa', 'af', 'ak', 'sq', 'am', 'ar', 'hy', 'as', 'ay'] ...


In [4]:
# 2. Download subtitles (Nepali 'ne') and save clean text
files = download_subtitles(video_url, languages=["ne", "ne-orig"], output_dir="downloads", clean_text=True)

[youtube] Extracting URL: https://www.youtube.com/watch?v=JgNZQ4ZTpZQ


[youtube] JgNZQ4ZTpZQ: Downloading webpage


[youtube] JgNZQ4ZTpZQ: Downloading visionos player API JSON


[youtube] JgNZQ4ZTpZQ: Downloading m3u8 information


[info] JgNZQ4ZTpZQ: Downloading subtitles: ne, ne-orig


...

[download]   63.00KiB at  316.66KiB/s (00:00:00)

[download]  127.00KiB at  433.04KiB/s (00:00:00)

[download]  198.45KiB at  570.05KiB/s (00:00:00)

[download] 100% of  198.45KiB in 00:00:00 at 286.01KiB/s

Clean transcript saved to: downloads/पत्रकारलाई ‘ठीक्क पर्‍यो’ भन्नेहरूले पनि बुझ्नुपर्ने चुरो कुरा ｜ Press Freedom Explained Nepal ｜_JgNZQ4ZTpZQ.ne.txt
Clean transcript saved to: downloads/पत्रकारलाई ‘ठीक्क पर्‍यो’ भन्नेहरूले पनि बुझ्नुपर्ने चुरो कुरा ｜ Press Freedom Explained Nepal ｜_JgNZQ4ZTpZQ.ne-orig.txt


In [5]:
# 3. Preview clean transcript
txt_files = list(Path("downloads").glob("*.txt"))
if txt_files:
    with open(txt_files[0], 'r', encoding='utf-8') as f:
        lines = f.readlines()
        print("".join(lines[:30]))

Video Title: पत्रकारलाई ‘ठीक्क पर्‍यो’ भन्नेहरूले पनि बुझ्नुपर्ने चुरो कुरा | Press Freedom Explained Nepal |
Video URL: https://www.youtube.com/watch?v=JgNZQ4ZTpZQ
----------------------------------------
नेपालका न्यज रुमले बन्दुक बोकेका राज्यका
सुरक्षाकर्मी पहिले पनि देखेका छन्। २०६१
माघ १९ मा तत्कालिन राजा ज्ञानेन्द्र
शाहले सत्ता हातमा लिएपछि सैनिकहरू देशका
प्रमुख पत्रिका रेडियो र टेलिभिजनका
कार्यालयमा छिरेका थिए। पत्रिकामा समाचार
छापिनु अघि सैनिक अधिकारीले हेर्थे।
सत्तालाई मन नपरेकाले नै रेडियोमा समाचार
रोकियो। राजनीतिक टिप्पणीमा प्रतिबन्ध
लगाइयो। कतिपय समाचार कक्षमै सेना बसेर के
छाप्ने के नछाप्ने भनेर निग्रानी गर्थ्यो।
त्यो राजाको प्रत्यक्ष शासनकाल थियो। आजको
नेपाल होइन। अहिले सेनाले सम्पादकको
कुर्सी छेउमा बसेर यो समाचार हाल, यो
समाचार नहाल भनेर आदेश दिइरहेको छैन।
राजनीतिक दल माथि प्रतिबन्ध छैन। इन्टरनेट
पनि बन्द गरिएको छैन। तर शुक्रबार साझ
काठमाडौँका एक पछि अर्को न्यज रुममा साधा
पोशाकका प्रहरी प्रवेश गरेको दृश्यले त्यो
इतिहास सम्झाएको छ। किनभने लोकतन्त्रमा
प्रश्न समाचार सहि थियो 

## 4. Downloading Playlist Subtitles (Batch Processing)

In [6]:
def download_playlist_subtitles(playlist_url, languages=["ne", "en"], output_dir="playlist_downloads", max_videos=None):
    """
    Harvest subtitles for an entire YouTube playlist using yt-dlp.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    out_tmpl = str(Path(output_dir) / "%(playlist_index)s_%(title)s_%(id)s.%(ext)s")

    ydl_opts = {
        'skip_download': True,
        'writesubtitles': True,
        'writeautomaticsub': True,
        'subtitleslangs': languages,
        'subtitlesformat': 'vtt',
        'outtmpl': out_tmpl,
        'quiet': False,
        'ignoreerrors': True,
        'playlistend': max_videos
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([playlist_url])

    # Clean all downloaded .vtt files
    for vtt_file in Path(output_dir).glob("*.vtt"):
        cleaned = clean_vtt_text(vtt_file)
        txt_path = vtt_file.with_suffix(".txt")
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(cleaned)
        print(f"Processed: {txt_path.name}")

    print("Playlist harvesting complete.")

# Example usage:
# download_playlist_subtitles("https://www.youtube.com/playlist?list=YOUR_PLAYLIST_ID", languages=["ne", "en"], max_videos=5)